# Laya multilíngue → ONNX INT8 para a API Fraus

Este notebook baixa a revisão fixada, exporta com o utilitário oficial do Laya, compara PyTorch, ONNX FP32 e ONNX INT8 e baixa um ZIP pronto para entrar no bundle externo da Vercel. Execute **Runtime > Run all**; não há seleção manual de arquivos.

In [ ]:
!pip -q install 'protobuf>=5.29.1,<6' laya==0.3.21 onnx>=1.17 onnxruntime>=1.20 onnxscript>=0.1 huggingface_hub>=0.27


In [ ]:
from pathlib import Path
from huggingface_hub import snapshot_download

REPO = 'convaiinnovations/laya'
REVISAO = '55cf4c4ebb4ebe31b2550e8bdf3bd21b99753851'
RAIZ = Path('/content/fraus-laya')
DESTINO = RAIZ / 'modelos-onnx' / 'laya-ironia'
DESTINO.mkdir(parents=True, exist_ok=True)
snapshot = Path(snapshot_download(REPO, revision=REVISAO, allow_patterns=['multilingual/*']))
MODELO = snapshot / 'multilingual'
print(MODELO)


In [ ]:
import shutil
from laya.agent import Agent
import torch

agente_torch = Agent(str(MODELO), compile=False, device='cpu')
entradas = (
    torch.randint(0, 100, (1, 16), dtype=torch.long),
    torch.ones((1, 16), dtype=torch.long),
    torch.tensor([[1, 5]], dtype=torch.long),
    torch.tensor([[True, True]], dtype=torch.bool),
    torch.tensor([0], dtype=torch.long),
)
fp32 = DESTINO / 'laya.onnx'
torch.onnx.export(
    agente_torch.model, entradas, str(fp32), export_params=True, opset_version=18,
    do_constant_folding=True,
    input_names=['input_ids','attention_mask','marker_pos','marker_mask','qtype'],
    output_names=['logits','act_logits'],
    dynamic_axes={
      'input_ids': {0:'batch_size',1:'seq_len'}, 'attention_mask': {0:'batch_size',1:'seq_len'},
      'marker_pos': {0:'batch_size',1:'num_markers'}, 'marker_mask': {0:'batch_size',1:'num_markers'},
      'qtype': {0:'batch_size'}, 'logits': {0:'batch_size',1:'num_markers'},
      'act_logits': {0:'batch_size'}},
)
shutil.copy2(MODELO/'rl_agent_config.json', DESTINO/'rl_agent_config.json')
shutil.copytree(MODELO/'tokenizer', DESTINO/'tokenizer', dirs_exist_ok=True)
shutil.copytree(MODELO/'encoder', DESTINO/'encoder', dirs_exist_ok=True)


In [ ]:
import onnx
from onnxruntime.quantization import QuantType, quantize_dynamic

modelo = onnx.load(str(fp32))
del modelo.graph.value_info[:]
int8 = DESTINO / 'laya.int8.candidato.onnx'
quantize_dynamic(model_input=modelo, model_output=str(int8), op_types_to_quantize=['MatMul'], weight_type=QuantType.QInt8, per_channel=True)
print(f'FP32: {fp32.stat().st_size/1024**2:.1f} MB | INT8: {int8.stat().st_size/1024**2:.1f} MB')


In [ ]:
from laya.onnx_agent import ONNXAgent

PERGUNTAS = {'ironia': {'type':'choice', 'instructions':'A fala do cliente é irônica? Considere incongruência entre o sentido literal e a situação descrita.', 'criteria': {'A':'há ironia ou incongruência entre elogio literal e situação negativa', 'B':'a fala é literal e não irônica'}}}
FRASES = [
 'ótimo serviço, só levou duas semanas para responder',
 'nossa, resolveu rapidinho, obrigado',
 'perfeito, cancelaram meu pedido sem avisar',
 'obrigado pelo atendimento rápido',
 'o pedido chegou hoje e está tudo certo',
]
agente_fp32 = ONNXAgent(str(DESTINO), onnx_path=str(fp32))
agente_int8 = ONNXAgent(str(DESTINO), onnx_path=str(int8))
def probs(agente):
    # O export oficial conserva batch 1 em operações internas; valide uma frase por sessão.
    return [agente.predict_batch([f], PERGUNTAS, lang='pt')[0]['answers']['ironia']['probabilities']['A'] for f in FRASES]
p_torch = [agente_torch.predict(f, PERGUNTAS, lang='pt')['answers']['ironia']['probabilities']['A'] for f in FRASES]
p_fp32, p_int8 = probs(agente_fp32), probs(agente_int8)
deriva_fp32 = max(abs(a-b) for a,b in zip(p_torch, p_fp32))
deriva = max(abs(a-b) for a,b in zip(p_torch, p_int8))
trocas = sum((a >= .5) != (b >= .5) for a,b in zip(p_torch, p_int8))
for frase, a, b, c in zip(FRASES, p_torch, p_fp32, p_int8): print(f'{a:.4f} | {b:.4f} | {c:.4f} | {frase}')
assert deriva_fp32 <= 1e-4, f'export FP32 divergiu do PyTorch em {deriva_fp32:.6f}'
# Um grafo INT8 so substitui o FP32 se nao depender dos pesos externos dele.
grafo_int8 = onnx.load(str(int8), load_external_data=False)
int8_independente = all(t.data_location != onnx.TensorProto.EXTERNAL for t in grafo_int8.graph.initializer)
int8_aprovado = trocas == 0 and deriva <= .10 and int8_independente
if int8_aprovado:
    int8.replace(fp32)
    (DESTINO/'laya.onnx.data').unlink(missing_ok=True)
    formato = 'int8'
else:
    int8.unlink()
    formato = 'fp32'
    print(f'INT8 recusado: {trocas} trocas, deriva {deriva:.4f}; mantendo FP32 validado.')
int8.with_name(int8.name + '.data').unlink(missing_ok=True)
print({'formato_escolhido': formato, 'trocas_int8': trocas, 'deriva_int8': deriva, 'deriva_fp32': deriva_fp32, 'int8_independente': int8_independente})


In [ ]:
import hashlib, json, zipfile
from google.colab import files

manifesto = {'schema':1, 'modelo':REPO, 'revisao':REVISAO, 'arquivo':'laya.onnx', 'formato':formato, 'sha256':hashlib.sha256(fp32.read_bytes()).hexdigest(), 'trocas_int8':trocas, 'deriva_int8':deriva, 'deriva_fp32':deriva_fp32}
(DESTINO/'manifesto.json').write_text(json.dumps(manifesto, ensure_ascii=False, indent=2)+'\n')
zip_saida = Path('/content/laya-ironia-onnx-validado.zip')
with zipfile.ZipFile(zip_saida, 'w', zipfile.ZIP_DEFLATED) as z:
    for arquivo in DESTINO.rglob('*'):
        if arquivo.is_file(): z.write(arquivo, arquivo.relative_to(RAIZ))
print(manifesto, zip_saida.stat().st_size)
files.download(str(zip_saida))
